In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [4]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

In [5]:
ds = ds.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 5372
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='Langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:01<00:00, 4901.97 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [9]:
tokenized_ds['train'][0]

{'input_ids': [26283,
  29,
  38566,
  1721,
  2043,
  6123,
  1277,
  975,
  10096,
  5040,
  33087,
  1518,
  189,
  1974,
  1817,
  1639,
  6724,
  672,
  189,
  4340,
  17245,
  29,
  210,
  1036,
  2153,
  2043,
  6123,
  420,
  9640,
  33087,
  31167,
  355,
  2290,
  2421,
  2043,
  6123,
  355,
  1032,
  5,
  1974,
  43749,
  1036,
  2153,
  355,
  12689,
  5,
  1974,
  5,
  3039,
  3496,
  13014,
  420,
  2],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [-100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  1036,
  2153,
  2043,
  6123,
  420,
  9640,
  33087,
  31167,
  355,
  2290,
  2421,
  2043,
  61

In [10]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

'Human: 写一篇简短的评论，评论电影《穿普拉达的女王》。\n\nAssistant: 《穿普拉达的女王》是一部精彩的电影，它带我们走进了时尚杂志的世界。影片通过安德莉亚的视角，展示了她如何在时尚界的顶尖杂志《Runway》打拼，并与杂志主编米兰达·普利斯特利的关系发生了变化。梅丽尔·斯特里普出演的米兰达一角非常出色，她完美地表现了这个角色的冷酷、严厉和智慧。除此之外，这部电影的服装造型也十分出彩，展示了时尚界的风采。总的来说，这是一部值得一看的电影，它不仅有着出色的演员阵容和精美的服装，还讲述了一个关于梦想、奋斗和选择的感人故事。</s>'

In [11]:
type(tokenized_ds['train'][1]["input_ids"]), type(tokenized_ds['train'][1]["labels"])

(list, list)

In [12]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][1]["labels"])))

'《穿普拉达的女王》是一部精彩的电影，它带我们走进了时尚杂志的世界。影片通过安德莉亚的视角，展示了她如何在时尚界的顶尖杂志《Runway》打拼，并与杂志主编米兰达·普利斯特利的关系发生了变化。梅丽尔·斯特里普出演的米兰达一角非常出色，她完美地表现了这个角色的冷酷、严厉和智慧。除此之外，这部电影的服装造型也十分出彩，展示了时尚界的风采。总的来说，这是一部值得一看的电影，它不仅有着出色的演员阵容和精美的服装，还讲述了一个关于梦想、奋斗和选择的感人故事。</s>'

In [13]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-1b4-zh", low_cpu_mem_usage=True)
model

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(46145, 2048)
    (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
          (dense): Linear(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (l

In [14]:
sum(param.numel() for param in model.parameters())

1303111680

In [15]:
# bitfit
# 选择模型参数里面的所有bias部分

num_param = 0
for name, param in model.named_parameters():
    if "bias" not in name:
        param.requires_grad = False
    else:
        num_param += param.numel()

num_param

544768

In [16]:
num_param / sum(param.numel() for param in model.parameters())

0.000418051659240749

In [17]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=50,
    eval_strategy='steps',
    num_train_epochs=1,
    gradient_checkpointing=True
)

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'].select(range(4000)),
    eval_dataset=tokenized_ds['test'].select(range(500)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

C:\Users\10433\AppData\Local\Temp\ipykernel_19124\2468238817.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [19]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
50,2.742600,2.748515
100,2.681700,2.652841
150,2.489000,2.598040
200,2.552300,2.554468
250,2.504700,2.524081
300,2.505600,2.507881
350,2.442900,2.494834
400,2.427800,2.487723
450,2.422200,2.479121
500,2.465800,2.472986


TrainOutput(global_step=1000, training_loss=2.471853370666504, metrics={'train_runtime': 931.8722, 'train_samples_per_second': 4.292, 'train_steps_per_second': 1.073, 'total_flos': 2188334751006720.0, 'train_loss': 2.471853370666504, 'epoch': 1.0})

In [20]:
model = model.cuda()
# 将模型切换到推理模式
model.eval()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
result = model.generate(**ipt, max_length=128, do_sample=True)[0]
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考生在备考考试时，要根据自己的知识水平和复习计划制定切实可行的复习计划。比如，针对自己薄弱环节，多读一些提高知识水平的相关书籍并自学相关技巧;同时，考生也可以利用学习班、培训机构的辅导或答疑模式来提升自己的学习效果。此外，考生也应保持良好的心态，积极面对考试。'

In [21]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [22]:
ipt = "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: "
pipe(ipt, max_length=256, do_sample=True)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧有很多，比如：\n（1） 备考过程中，一定提前设计好自己的复习计划，合理安排时间，避免盲目备考。 \n（2） 合理安排复习进度，不要心猿意马，在复习进度上出现偏差，比如复习进度过快、过慢等，都容易影响考试成绩。\n（3） 考前不要盲目复习，要根据自己的实际情况，合理安排复习计划，将自己擅长、擅长的科目进行重点复习，以达到事半功倍的效果。\n（4） 考前一定做好充分心理准备，做好充足的心理保障，不要因为压力过大、焦虑不安等负面情绪影响考试发挥。\n（5） 考前一定要合理安排作息，保证充足的睡眠，这样才能保证考试时发挥最佳水平。'}]